In [1]:
import sys

sys.path.append("..")

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from torch import nn

from src.models import (
    ImageEncoder,
    MultimodalClassifier,
    TextEncoder,
)
from src.preprocessing import (
    ImageProcessor,
    TextProcessor,
)
from src.utils.config import load_config
from src.utils.device import get_device

PROJECT_ROOT = Path.cwd().parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"

config = load_config(CONFIG_PATH)

device = get_device(
    config["training"]["device"]
)

print("Device:", device)

c:\Users\AYUSH PRASAD\Desktop\Multimodal Misinformation Detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Loaing a small real batch

In [3]:
processed_dir = (
    PROJECT_ROOT
    / config["paths"]["processed_dir"]
)

train_df = pd.read_csv(
    processed_dir / "train.csv"
)

sample_df = train_df.head(4).copy()

display(
    sample_df[
        ["id", "text", "image_path", "label"]
    ]
)

,id,text,image_path,label
0,4jj0fh,funny dog names you havent thought of,images/train/4jj0fh.jpg,0
1,7tykik,destyler numb,images/train/7tykik.jpg,0
2,1jfqnp,us student abandoned in san diego jail gets m ...,images/train/1jfqnp.jpg,1
3,boa5ua,this horrified styrofoam,images/train/boa5ua.jpg,0


## Initializing both processors

In [4]:
text_config = config["models"]["text"]

text_processor = TextProcessor(
    model_name=text_config["name"],
    max_length=text_config["max_length"],
)

image_processor = ImageProcessor()

## Prepare text inputs

In [5]:
texts = sample_df["text"].tolist()

encoded = text_processor(texts)

encoded = {
    key: value.to(device)
    for key, value in encoded.items()
}

print("Text input shapes:")

for key, value in encoded.items():
    print(
        f"{key:15s}",
        tuple(value.shape),
        value.device,
    )

Text input shapes:
input_ids       (4, 13) cuda:0
attention_mask  (4, 13) cuda:0


## Prepare image inputs

In [6]:
image_paths = [
    processed_dir / path
    for path in sample_df["image_path"]
]

images = torch.stack(
    [
        image_processor(path)
        for path in image_paths
    ]
).to(device)

print("Image batch:", tuple(images.shape))
print("Device:", images.device)

Image batch: (4, 3, 224, 224)
Device: cuda:0


## Initialize the pretrained encoders

In [7]:
text_encoder = TextEncoder(
    model_name=text_config["name"],
    freeze=text_config["freeze"],
).to(device)

image_encoder = ImageEncoder(
    freeze=config["models"]["image"]["freeze"]
).to(device)

print(
    "Text embedding:",
    text_encoder.embedding_dim,
)

print(
    "Image embedding:",
    image_encoder.embedding_dim,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 33180.16it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text embedding: 768
Image embedding: 512


## Initialize the fusion model

In [8]:
fusion_config = config["models"]["fusion"]

model = MultimodalClassifier(
    text_encoder=text_encoder,
    image_encoder=image_encoder,
    hidden_dim=fusion_config["hidden_dim"],
    dropout=fusion_config["dropout"],
    num_classes=config["project"]["num_classes"],
    freeze_backbones=True,
).to(device)

print(model)

MultimodalClassifier(
  (text_encoder): TextEncoder(
    (model): DistilBertModel(
      (embeddings): Embeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (transformer): Transformer(
        (layer): ModuleList(
          (0-5): 6 x TransformerBlock(
            (attention): DistilBertSelfAttention(
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (ffn): FFN(
    

In [9]:
print("Text dimension :", model.text_dim)
print("Image dimension:", model.image_dim)
print("Fusion dimension:", model.fusion_dim)

Text dimension : 768
Image dimension: 512
Fusion dimension: 1280


## Multimodal Forward Pass

In [10]:
with torch.inference_mode():
    logits = model(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
        images=images,
    )

print("Logits shape:", tuple(logits.shape))
print("Logits dtype:", logits.dtype)
print("Device:", logits.device)

Logits shape: (4, 2)
Logits dtype: torch.float32
Device: cuda:0


## Testing probabilities and predictions

In [11]:
probabilities = torch.softmax(
    logits,
    dim=1,
)

predictions = torch.argmax(
    probabilities,
    dim=1,
)

print("Probabilities:")
print(probabilities)

print("\nPredictions:")
print(predictions)

Probabilities:
tensor([[0.3697, 0.6303],
        [0.4299, 0.5701],
        [0.4461, 0.5539],
        [0.3917, 0.6083]], device='cuda:0')

Predictions:
tensor([1, 1, 1, 1], device='cuda:0')


## Verifying that only the fusion classifier is trainable

In [12]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "Total parameters:    ",
    f"{total_parameters:,}",
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}",
)

Total parameters:     77,867,842
Trainable parameters: 328,450


## Verifying train/eval behavior

In [13]:
model.train()

print(
    "Text encoder training mode :",
    model.text_encoder.training,
)

print(
    "Image encoder training mode:",
    model.image_encoder.training,
)

print(
    "Classifier training mode   :",
    model.classifier.training,
)

Text encoder training mode : False
Image encoder training mode: False
Classifier training mode   : True


## Verifying gradient flow

In [14]:
labels = torch.tensor(
    sample_df["label"].tolist(),
    dtype=torch.long,
    device=device,
)

In [15]:
criterion = nn.CrossEntropyLoss()

model.train()

logits = model(
    input_ids=encoded["input_ids"],
    attention_mask=encoded["attention_mask"],
    images=images,
)

loss = criterion(
    logits,
    labels,
)

loss.backward()

print("Loss:", loss.item())

Loss: 0.7660920023918152


In [16]:
text_grads = [
    parameter.grad
    for parameter in model.text_encoder.parameters()
    if parameter.requires_grad
]

image_grads = [
    parameter.grad
    for parameter in model.image_encoder.parameters()
    if parameter.requires_grad
]

fusion_grads = [
    parameter.grad
    for parameter in model.classifier.parameters()
]

print(
    "Trainable text parameters:",
    len(text_grads),
)

print(
    "Trainable image parameters:",
    len(image_grads),
)

print(
    "Fusion parameters with gradients:",
    sum(
        gradient is not None
        for gradient in fusion_grads
    ),
)

Trainable text parameters: 0
Trainable image parameters: 0
Fusion parameters with gradients: 4


## Final sanity check

In [17]:
assert model.text_dim == 768
assert model.image_dim == 512
assert model.fusion_dim == 1280

assert logits.shape == (4, 2)

assert trainable_parameters == 328_450

assert model.text_encoder.training is False
assert model.image_encoder.training is False
assert model.classifier.training is True

assert all(
    parameter.grad is None
    for parameter in model.text_encoder.parameters()
)

assert all(
    parameter.grad is None
    for parameter in model.image_encoder.parameters()
)

assert all(
    parameter.grad is not None
    for parameter in model.classifier.parameters()
)

print("Multimodal fusion sanity checks: PASSED")

Multimodal fusion sanity checks: PASSED


```
                    INPUT
                   /     \
                  /       \
              TEXT        IMAGE
               │            │
          DistilBERT     ResNet-18
               │            │
             768-D        512-D
               │            │
               └─────┬──────┘
                     │
                Concatenate
                     │
                  1280-D
                     │
               Linear 256
                     │
                    ReLU
                     │
                Dropout 0.30
                     │
                Linear 2
                     │
                   Logits
                  /      \
               Real      Fake
```